In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt
import pickle

ROOT = '/content/drive/MyDrive/GAMEEMO'
SAVE_DIR = '/content/drive/MyDrive/GAMEEMO_processed'
os.makedirs(SAVE_DIR, exist_ok=True)

fs = 128  # GAMEEMO sampling rate

In [ ]:
# label mapping based on the four games
# G1 = train sim (boring), G2 = unravel (calm), G3 = slender (horror), G4 = goat sim (funny)
game_to_label = {
    'G1': 0,  # boring
    'G2': 1,  # calm
    'G3': 2,  # horror
    'G4': 3,  # funny
}

In [ ]:
def bandpass_filter(signal, low=0.5, high=45, fs=128, order=4):
    nyquist = fs / 2
    high = min(high, nyquist - 1)  # safety margin
    b, a = butter(order, [low / nyquist, high / nyquist], btype='band')
    return filtfilt(b, a, signal, axis=0)

In [ ]:
# find every preprocessed csv file across all subject folders
# using recursive glob so the slightly inconsistent nesting (S26 inside S28) doesn't break things
pattern = os.path.join(ROOT, '**', 'Preprocessed EEG Data', '.csv format', '*AllChannels.csv')
all_files = glob.glob(pattern, recursive=True)

print('total preprocessed csv files found:', len(all_files))

total preprocessed csv files found: 112


In [ ]:
# quick check on one file before processing everything, just to confirm column structure
sample_df = pd.read_csv(all_files[0])
print(sample_df.shape)
print(sample_df.columns.tolist())
sample_df.head()

(38252, 15)
['AF3', 'AF4', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8', 'Unnamed: 14']


,AF3,AF4,F3,F4,F7,F8,FC5,FC6,O1,O2,P7,P8,T7,T8,Unnamed: 14
0,158.3537,121.2361,-75.9229,-24.3436,200.7745,194.0255,-31.5745,11.3282,-32.5383,-11.3282,-29.1639,-261.5128,47.0000,47.0000,NaN
1,158.0118,120.7106,-76.1880,-20.9546,190.6559,185.2763,-28.7159,11.6126,-31.0682,-11.6126,-20.1834,-247.2683,45.1441,47.5542,NaN
2,150.4593,114.9138,-73.5449,-19.6973,182.1091,174.6421,-28.9211,7.0595,-27.2759,-7.0595,-20.9006,-233.8782,55.9328,43.7371,NaN
3,146.7341,108.5011,-66.7223,-22.3721,181.7876,168.5017,-25.7398,3.7436,-27.5678,-3.7436,-22.0569,-223.7021,64.1462,44.0052,NaN
4,136.9660,95.2424,-54.0421,-21.9938,163.6498,152.6072,-17.9287,4.0011,-21.0932,-4.0011,-15.9130,-205.4593,58.3690,44.7391,NaN


In [ ]:
window_sec = 4
window_len = window_sec * fs
step = window_len  # non overlapping windows

all_segments = []
all_labels = []
all_subject_ids = []

seen_subject_game = set()  # avoid double counting the duplicated S26 folder

for filepath in all_files:
    filename = os.path.basename(filepath)

    match = re.match(r'S(\d+)(G\d)AllChannels\.csv', filename)
    if not match:
        print('skipping unexpected filename:', filename)
        continue

    subject_id = int(match.group(1))
    game_code = match.group(2)

    key = (subject_id, game_code)
    if key in seen_subject_game:
        continue  # already processed this subject/game combo from another folder copy
    seen_subject_game.add(key)

    label = game_to_label[game_code]

    df = pd.read_csv(filepath)

    # keep only numeric columns, drop anything like a timestamp or label column if present
    numeric_df = df.select_dtypes(include=[np.number])

    # GAMEEMO has 14 EEG channels, if extra numeric columns exist (like a sample index)
    # they tend to be monotonically increasing, so we drop any column that is just a counter
    cols_to_drop = []
    for col in numeric_df.columns:
        values = numeric_df[col].values
        if np.all(np.diff(values) >= 0) and values[0] in (0, 1):
            cols_to_drop.append(col)
    numeric_df = numeric_df.drop(columns=cols_to_drop)

    signal = numeric_df.values.astype(np.float32)

    if signal.shape[1] < 14:
        print('warning, fewer than 14 channels found for', filename, 'shape:', signal.shape)
    elif signal.shape[1] > 14:
        signal = signal[:, :14]  # keep first 14 channels only

    filtered = bandpass_filter(signal, fs=fs)

    mean = filtered.mean(axis=0, keepdims=True)
    std = filtered.std(axis=0, keepdims=True)
    std[std == 0] = 1
    normalized = (filtered - mean) / std

    n_windows = normalized.shape[0] // window_len
    for w in range(n_windows):
        start = w * step
        end = start + window_len
        segment = normalized[start:end, :]
        all_segments.append(segment)
        all_labels.append(label)
        all_subject_ids.append(subject_id)

print('finished reading all subjects')

finished reading all subjects


In [ ]:
all_segments = np.array(all_segments, dtype=np.float32)
all_labels = np.array(all_labels, dtype=np.int64)
all_subject_ids = np.array(all_subject_ids, dtype=np.int64)

print('final segment array shape:', all_segments.shape)
print('final label array shape:', all_labels.shape)
print('unique subjects processed:', len(np.unique(all_subject_ids)))
print('label distribution:', np.bincount(all_labels))

final segment array shape: (8288, 512, 14)
final label array shape: (8288,)
unique subjects processed: 28
label distribution: [2072 2072 2072 2072]


In [ ]:
output = {
    'X': all_segments,
    'y': all_labels,
    'subject_ids': all_subject_ids,
    'fs': fs,
    'window_sec': window_sec,
    'channels': 14,
    'label_names': ['boring', 'calm', 'horror', 'funny']
}

save_path = os.path.join(SAVE_DIR, 'gameemo_preprocessed.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(output, f)

print('saved preprocessed data to', save_path)

saved preprocessed data to /content/drive/MyDrive/GAMEEMO_processed/gameemo_preprocessed.pkl
